# Task 2 — Run on H100 (Colab / remote GPU)

This notebook is a hardened version of the Task 2 runner:

- Clones the repo into `/content` without creating nested folders.
- Auto-detects `REPO_ROOT` (must contain `scripts/task2/train.py`).
- Mounts Drive and verifies `DATA_ROOT` contains `train.jsonl`, `val.jsonl`, `spm.model`.
- Uses `python -m scripts.task2.*` to run the `.py` training code.

## Before you start

- Runtime → Change runtime type → **GPU** (A100).
- Ensure Drive folder `MyDrive/svg_task2_data/` contains:
  - `train.jsonl`
  - `val.jsonl`
  - `spm.model`


In [1]:
# Fresh clone into /content (avoids nested repo folders)
%cd /content
!rm -rf svg-scaling-laws-transformers
!git clone https://github.com/yuelihe2-svg/svg-scaling-laws-transformers.git

import os

# Resolve the repo root automatically
CANDIDATES = [
    "/content/svg-scaling-laws-transformers",
    "/content/svg-scaling-laws-transformers/svg-scaling-laws-transformers",
]
REPO_ROOT = None
for root in CANDIDATES:
    p = os.path.join(root, "scripts", "task2", "train.py")
    print(root, "->", os.path.exists(p))
    if REPO_ROOT is None and os.path.exists(p):
        REPO_ROOT = root

assert REPO_ROOT is not None, "Could not find scripts/task2/train.py in expected locations"
print("Using REPO_ROOT =", REPO_ROOT)
%cd $REPO_ROOT
!pwd
!ls scripts/task2


/content
Cloning into 'svg-scaling-laws-transformers'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 57 (delta 15), reused 46 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 39.56 KiB | 2.47 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/svg-scaling-laws-transformers -> True
/content/svg-scaling-laws-transformers/svg-scaling-laws-transformers -> False
Using REPO_ROOT = /content/svg-scaling-laws-transformers
/content/svg-scaling-laws-transformers
/content/svg-scaling-laws-transformers
config_presets.py  __init__.py	model.py	 README.md
data.py		   lr_sweep.py	plot_scaling.py  train.py


In [2]:
# Install deps (Colab already has some; this is safe)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'numpy', 'matplotlib', 'scipy', 'tqdm'])

# Torch: if needed, install CUDA wheel (adjust if your runtime uses a different CUDA)
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', '--index-url', 'https://download.pytorch.org/whl/cu124'])

import torch
print('torch', torch.__version__, 'cuda_available=', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


torch 2.10.0+cu128 cuda_available= True
gpu: NVIDIA A100-SXM4-40GB


In [3]:
# Mount Drive and verify data
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/svg_task2_data"
assert os.path.exists(f"{DATA_ROOT}/train.jsonl"), "train.jsonl not found in DATA_ROOT"
assert os.path.exists(f"{DATA_ROOT}/val.jsonl"), "val.jsonl not found in DATA_ROOT"
assert os.path.exists(f"{DATA_ROOT}/spm.model"), "spm.model not found in DATA_ROOT"
print("Using DATA_ROOT =", DATA_ROOT)
!ls -lh "$DATA_ROOT"


Mounted at /content/drive
Using DATA_ROOT = /content/drive/MyDrive/svg_task2_data
total 436M
drwx------ 2 root root 4.0K Apr 21 15:35 outputs_task2_backup_20260421_153519
drwx------ 2 root root 4.0K Apr 21 16:35 outputs_task2_backup_20260421_163526
drwx------ 2 root root 4.0K Apr 21 16:50 outputs_task2_backup_20260421_165057
drwx------ 2 root root 4.0K Apr 21 19:11 outputs_task2_backup_20260421_191122
-rw------- 1 root root 290K Apr 21 02:08 spm.model
-rw------- 1 root root 432M Apr 21 02:11 train.jsonl
-rw------- 1 root root 4.4M Apr 21 02:11 val.jsonl


## Step A — LR sweep on Tiny

Run once, then set `BEST_LR` to the best value in `outputs/task2/lr_sweep.csv`.

In [4]:
!python -m scripts.task2.lr_sweep \
  --train-jsonl {DATA_ROOT}/train.jsonl \
  --val-jsonl {DATA_ROOT}/val.jsonl \
  --spm-model {DATA_ROOT}/spm.model \
  --lrs 1e-4 3e-4 1e-3 3e-3 1e-2 3e-2 1e-1 \
  --out-csv outputs/task2/lr_sweep.csv


Running: /usr/bin/python3 -m scripts.task2.train --train-jsonl /content/drive/MyDrive/svg_task2_data/train.jsonl --val-jsonl /content/drive/MyDrive/svg_task2_data/val.jsonl --spm-model /content/drive/MyDrive/svg_task2_data/spm.model --preset tiny --lr 0.0001 --tokens-per-batch 32768 --block-size 512 --warmup-steps 2000 --out-dir outputs/task2/sweep_tiny_lr0.0001
Loading train stream ...
Loading val stream ...
Train tokens (incl. EOS): 103,255,934 | Val: 1,052,909
Preset=tiny ~~1M | Actual trainable params: 1,761,792
batch_size=64 block_size=512 tokens/step=32768
steps_per_epoch=3151 epochs=1 -> max_steps=3151
step 0/3151 train_loss=8.1766 lr=5.00e-08 tok/s=40893
step 50/3151 train_loss=8.1061 lr=2.55e-06 tok/s=3492487
step 100/3151 train_loss=7.9385 lr=5.05e-06 tok/s=3652685
step 150/3151 train_loss=7.7611 lr=7.55e-06 tok/s=3013096
step 200/3151 train_loss=7.5713 lr=1.01e-05 tok/s=3452669
step 250/3151 train_loss=7.3493 lr=1.26e-05 tok/s=3507525
step 300/3151 train_loss=7.1171 lr=1.51e

## Step B — Train each model size (1 epoch)

- Keep `TOKENS_PER_BATCH` and `BLOCK_SIZE` the **same across all models**.
- If you change them due to OOM, change them for **all models** (or re-run earlier ones).


In [6]:
BEST_LR = 0.01  # set from lr_sweep.csv
TOKENS_PER_BATCH = 32768
BLOCK_SIZE = 512


In [12]:
# tiny
!python -m scripts.task2.train \
  --train-jsonl {DATA_ROOT}/train.jsonl \
  --val-jsonl {DATA_ROOT}/val.jsonl \
  --spm-model {DATA_ROOT}/spm.model \
  --preset tiny \
  --lr {BEST_LR} \
  --tokens-per-batch {TOKENS_PER_BATCH} \
  --block-size {BLOCK_SIZE} \
  --out-dir outputs/task2/tiny \
  --device cuda


Loading train stream ...
Loading val stream ...
Train tokens (incl. EOS): 103,255,934 | Val: 1,052,909
Preset=tiny ~~1M | Actual trainable params: 1,761,792
batch_size=64 block_size=512 tokens/step=32768
steps_per_epoch=3151 epochs=1 -> max_steps=3151
step 0/3151 train_loss=8.1766 lr=5.00e-06 tok/s=133691
step 50/3151 train_loss=6.3951 lr=2.55e-04 tok/s=3009951
step 100/3151 train_loss=4.2915 lr=5.05e-04 tok/s=3550601
step 150/3151 train_loss=3.0980 lr=7.55e-04 tok/s=3246739
step 200/3151 train_loss=2.8779 lr=1.01e-03 tok/s=3273884
step 250/3151 train_loss=2.7945 lr=1.26e-03 tok/s=3351027
step 300/3151 train_loss=2.5761 lr=1.51e-03 tok/s=3599939
step 350/3151 train_loss=2.3334 lr=1.76e-03 tok/s=3519954
step 400/3151 train_loss=2.2778 lr=2.00e-03 tok/s=3505863
step 450/3151 train_loss=2.2000 lr=2.26e-03 tok/s=3389393
step 500/3151 train_loss=2.1128 lr=2.50e-03 tok/s=3623013
step 550/3151 train_loss=2.0451 lr=2.76e-03 tok/s=3379651
step 600/3151 train_loss=2.0547 lr=3.00e-03 tok/s=341162

In [11]:
# small
!python -m scripts.task2.train \
  --train-jsonl {DATA_ROOT}/train.jsonl \
  --val-jsonl {DATA_ROOT}/val.jsonl \
  --spm-model {DATA_ROOT}/spm.model \
  --preset small \
  --lr {BEST_LR} \
  --tokens-per-batch {TOKENS_PER_BATCH} \
  --block-size {BLOCK_SIZE} \
  --out-dir outputs/task2/small \
  --device cuda


Loading train stream ...
Loading val stream ...
Train tokens (incl. EOS): 103,255,934 | Val: 1,052,909
Preset=small ~~3M | Actual trainable params: 4,120,704
batch_size=64 block_size=512 tokens/step=32768
steps_per_epoch=3151 epochs=1 -> max_steps=3151
step 0/3151 train_loss=8.2262 lr=5.00e-06 tok/s=118037
step 50/3151 train_loss=5.5211 lr=2.55e-04 tok/s=2341732
step 100/3151 train_loss=3.3501 lr=5.05e-04 tok/s=2168402
step 150/3151 train_loss=2.9140 lr=7.55e-04 tok/s=2460160
step 200/3151 train_loss=2.7181 lr=1.01e-03 tok/s=2273722
step 250/3151 train_loss=2.5790 lr=1.26e-03 tok/s=2422522
step 300/3151 train_loss=2.3068 lr=1.51e-03 tok/s=2367814
step 350/3151 train_loss=2.1337 lr=1.76e-03 tok/s=2280467
step 400/3151 train_loss=2.1030 lr=2.00e-03 tok/s=2286621
step 450/3151 train_loss=2.0672 lr=2.26e-03 tok/s=2407228
step 500/3151 train_loss=2.0012 lr=2.50e-03 tok/s=2329901
step 550/3151 train_loss=1.9412 lr=2.76e-03 tok/s=2354341
step 600/3151 train_loss=1.9304 lr=3.00e-03 tok/s=23499

In [10]:
# medium
!python -m scripts.task2.train \
  --train-jsonl {DATA_ROOT}/train.jsonl \
  --val-jsonl {DATA_ROOT}/val.jsonl \
  --spm-model {DATA_ROOT}/spm.model \
  --preset medium \
  --lr {BEST_LR} \
  --tokens-per-batch {TOKENS_PER_BATCH} \
  --block-size {BLOCK_SIZE} \
  --out-dir outputs/task2/medium \
  --device cuda


Loading train stream ...
Loading val stream ...
Train tokens (incl. EOS): 103,255,934 | Val: 1,052,909
Preset=medium ~~10M | Actual trainable params: 13,549,824
batch_size=64 block_size=512 tokens/step=32768
steps_per_epoch=3151 epochs=1 -> max_steps=3151
step 0/3151 train_loss=8.2729 lr=5.00e-06 tok/s=89527
step 50/3151 train_loss=3.6763 lr=2.55e-04 tok/s=2325726
step 100/3151 train_loss=2.9737 lr=5.05e-04 tok/s=2076018
step 150/3151 train_loss=2.6229 lr=7.55e-04 tok/s=2310319
step 200/3151 train_loss=2.3627 lr=1.01e-03 tok/s=2308832
step 250/3151 train_loss=2.2018 lr=1.26e-03 tok/s=2106758
step 300/3151 train_loss=2.0999 lr=1.51e-03 tok/s=2341826
step 350/3151 train_loss=1.9647 lr=1.76e-03 tok/s=2318441
step 400/3151 train_loss=1.9393 lr=2.00e-03 tok/s=2358738
step 450/3151 train_loss=1.9363 lr=2.26e-03 tok/s=2336097
step 500/3151 train_loss=1.8491 lr=2.50e-03 tok/s=2358522
step 550/3151 train_loss=1.8255 lr=2.76e-03 tok/s=2085102
step 600/3151 train_loss=1.8204 lr=3.00e-03 tok/s=234

In [9]:
# large
!python -m scripts.task2.train \
  --train-jsonl {DATA_ROOT}/train.jsonl \
  --val-jsonl {DATA_ROOT}/val.jsonl \
  --spm-model {DATA_ROOT}/spm.model \
  --preset large \
  --lr {BEST_LR} \
  --tokens-per-batch {TOKENS_PER_BATCH} \
  --block-size {BLOCK_SIZE} \
  --out-dir outputs/task2/large \
  --device cuda


Loading train stream ...
Loading val stream ...
Train tokens (incl. EOS): 103,255,934 | Val: 1,052,909
Preset=large ~~30M | Actual trainable params: 35,386,368
batch_size=64 block_size=512 tokens/step=32768
steps_per_epoch=3151 epochs=1 -> max_steps=3151
step 0/3151 train_loss=8.3185 lr=5.00e-06 tok/s=51273
step 50/3151 train_loss=3.3136 lr=2.55e-04 tok/s=1516603
step 100/3151 train_loss=2.8858 lr=5.05e-04 tok/s=1335081
step 150/3151 train_loss=2.5727 lr=7.55e-04 tok/s=1534961
step 200/3151 train_loss=2.2544 lr=1.01e-03 tok/s=1375689
step 250/3151 train_loss=2.1682 lr=1.26e-03 tok/s=1540014
step 300/3151 train_loss=2.0662 lr=1.51e-03 tok/s=1484250
step 350/3151 train_loss=1.9686 lr=1.76e-03 tok/s=1550280
step 400/3151 train_loss=1.9925 lr=2.00e-03 tok/s=1550539
step 450/3151 train_loss=1.9304 lr=2.26e-03 tok/s=1334342
step 500/3151 train_loss=1.8423 lr=2.50e-03 tok/s=1526362
step 550/3151 train_loss=1.8620 lr=2.76e-03 tok/s=1533213
step 600/3151 train_loss=1.8024 lr=3.00e-03 tok/s=1552

In [7]:
# xl
!python -m scripts.task2.train \
  --train-jsonl {DATA_ROOT}/train.jsonl \
  --val-jsonl {DATA_ROOT}/val.jsonl \
  --spm-model {DATA_ROOT}/spm.model \
  --preset xl \
  --lr {BEST_LR} \
  --tokens-per-batch {TOKENS_PER_BATCH} \
  --block-size {BLOCK_SIZE} \
  --out-dir outputs/task2/xl \
  --device cuda


Loading train stream ...
Loading val stream ...
Train tokens (incl. EOS): 103,255,934 | Val: 1,052,909
Preset=xl ~~88M | Actual trainable params: 90,842,112
batch_size=64 block_size=512 tokens/step=32768
steps_per_epoch=3151 epochs=1 -> max_steps=3151
step 0/3151 train_loss=8.2546 lr=5.00e-06 tok/s=25218
step 50/3151 train_loss=3.0888 lr=2.55e-04 tok/s=1319192
step 100/3151 train_loss=2.8283 lr=5.05e-04 tok/s=1370573
step 150/3151 train_loss=2.3820 lr=7.55e-04 tok/s=1285975
step 200/3151 train_loss=2.1553 lr=1.01e-03 tok/s=1154313
step 250/3151 train_loss=2.0984 lr=1.26e-03 tok/s=1358126
step 300/3151 train_loss=2.1153 lr=1.51e-03 tok/s=1255537
step 350/3151 train_loss=1.9429 lr=1.76e-03 tok/s=1306752
step 400/3151 train_loss=1.9300 lr=2.00e-03 tok/s=1300142
step 450/3151 train_loss=1.9295 lr=2.26e-03 tok/s=1218930
step 500/3151 train_loss=1.8374 lr=2.50e-03 tok/s=1369769
step 550/3151 train_loss=1.7850 lr=2.76e-03 tok/s=1330560
step 600/3151 train_loss=1.7785 lr=3.00e-03 tok/s=1357530

## Step C — Backup results to Drive (run after each model)


In [13]:
!cp -r outputs/task2 "/content/drive/MyDrive/svg_task2_data/outputs_task2_backup_$(date +%Y%m%d_%H%M%S)"

